# 🏦 Credit Risk Intelligence Platform
### End-to-End ML Pipeline with Explainability, Calibration & Business Metrics

**Dataset:** Home Credit Default Risk (Kaggle)

**Skills Demonstrated:**
- Advanced feature engineering on real financial data
- Class imbalance handling (SMOTE + class weights)
- Model comparison with business-aware metrics
- Hyperparameter tuning (Optuna)
- SHAP explainability
- Probability calibration
- Cost-sensitive threshold optimization
- Professional model serialization

---
**Runtime:** ~10-12 min on Colab Free | **RAM:** <6GB | **GPU:** Not required

In [ ]:

!pip install shap optuna imbalanced-learn lightgbm scikit-learn pandas numpy matplotlib seaborn joblib kaggle --quiet

## 📥 Step 2: Download Dataset

We use the **Home Credit Default Risk** dataset from Kaggle.

**Option A (Kaggle API — recommended):**
Upload your `kaggle.json` API key, then run the cell below.

**Option B (Manual):** Download `application_train.csv` from https://www.kaggle.com/c/home-credit-default-risk/data and upload to Colab.


In [ ]:
from google.colab import files
files.upload()  # upload your kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c home-credit-default-risk -f application_train.csv
!unzip application_train.csv.zip

df_raw = pd.read_csv('application_train.csv')
print(df_raw.shape)  # should be (307511, 122)

In [ ]:
print(f"Target distribution:\n{df_raw['TARGET'].value_counts(normalize=True).round(3)}")


## 🔍 Step 3: Exploratory Data Analysis


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d2e'
plt.rcParams['axes.labelcolor'] = '#e0e0e0'
plt.rcParams['text.color'] = '#e0e0e0'
plt.rcParams['xtick.color'] = '#aaaaaa'
plt.rcParams['ytick.color'] = '#aaaaaa'
plt.rcParams['grid.color'] = '#2a2d3e'
plt.rcParams['axes.edgecolor'] = '#2a2d3e'

fig = plt.figure(figsize=(18, 12))
fig.suptitle('Credit Risk — Exploratory Data Analysis', fontsize=18, fontweight='bold',
             color='white', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

# 1. Target distribution
ax1 = fig.add_subplot(gs[0, 0])
target_counts = df_raw['TARGET'].value_counts()
colors = ['#4CAF50', '#FF5252']
bars = ax1.bar(['No Default', 'Default'], target_counts.values, color=colors, alpha=0.85, edgecolor='none')
for bar, val in zip(bars, target_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{val:,}\n({val/len(df_raw)*100:.1f}%)', ha='center', va='bottom',
             fontsize=10, color='white')
ax1.set_title('Class Imbalance', fontweight='bold', color='white')
ax1.set_ylabel('Count', color='#aaaaaa')

# 2. Age distribution by target
ax2 = fig.add_subplot(gs[0, 1])
age = (-df_raw['DAYS_BIRTH'] / 365).clip(18, 70)
for t, color, label in zip([0, 1], ['#4CAF50', '#FF5252'], ['No Default', 'Default']):
    mask = df_raw['TARGET'] == t
    ax2.hist(age[mask], bins=30, alpha=0.6, color=color, label=label, density=True)
ax2.set_title('Age Distribution by Default', fontweight='bold', color='white')
ax2.set_xlabel('Age (years)')
ax2.legend(fontsize=9)

# 3. Credit amount distribution
ax3 = fig.add_subplot(gs[0, 2])
for t, color, label in zip([0, 1], ['#4CAF50', '#FF5252'], ['No Default', 'Default']):
    mask = df_raw['TARGET'] == t
    vals = np.log1p(df_raw.loc[mask, 'AMT_CREDIT'].dropna())
    ax3.hist(vals, bins=30, alpha=0.6, color=color, label=label, density=True)
ax3.set_title('Log(Credit Amount) by Default', fontweight='bold', color='white')
ax3.set_xlabel('Log(AMT_CREDIT)')
ax3.legend(fontsize=9)

# 4. Missing value heatmap (top 15 cols)
ax4 = fig.add_subplot(gs[1, 0])
missing = df_raw.isnull().mean().sort_values(ascending=False).head(15)
if missing.sum() > 0:
    colors_missing = ['#FF5252' if v > 0.3 else '#FFB74D' if v > 0.1 else '#66BB6A'
                      for v in missing.values]
    ax4.barh(range(len(missing)), missing.values * 100, color=colors_missing, alpha=0.85)
    ax4.set_yticks(range(len(missing)))
    ax4.set_yticklabels([c[:20] for c in missing.index], fontsize=8)
    ax4.set_title('Missing Values (%)', fontweight='bold', color='white')
    ax4.set_xlabel('Missing %')
else:
    ax4.text(0.5, 0.5, 'No missing values', ha='center', va='center', color='white')
    ax4.set_title('Missing Values', fontweight='bold', color='white')

# 5. EXT_SOURCE correlation with target
ax5 = fig.add_subplot(gs[1, 1])
ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in df_raw.columns]
if ext_cols:
    ext_data = [df_raw[c].dropna() for c in ext_cols]
    bp = ax5.boxplot(ext_data, labels=[c.replace('EXT_SOURCE_', 'EXT_') for c in ext_cols],
                     patch_artist=True,
                     boxprops=dict(facecolor='#1E90FF', alpha=0.7),
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='#aaaaaa'),
                     capprops=dict(color='#aaaaaa'),
                     flierprops=dict(marker='.', color='#FF5252', alpha=0.3))
ax5.set_title('External Score Distributions', fontweight='bold', color='white')
ax5.set_ylabel('Score')

# 6. Default rate by education
ax6 = fig.add_subplot(gs[1, 2])
if 'NAME_EDUCATION_TYPE' in df_raw.columns:
    edu_default = df_raw.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values(ascending=False)
    edu_labels = [e[:20] for e in edu_default.index]
    colors_edu = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(edu_default)))
    ax6.barh(range(len(edu_default)), edu_default.values * 100, color=colors_edu, alpha=0.85)
    ax6.set_yticks(range(len(edu_default)))
    ax6.set_yticklabels(edu_labels, fontsize=8)
    ax6.set_title('Default Rate by Education', fontweight='bold', color='white')
    ax6.set_xlabel('Default Rate (%)')

plt.savefig('/content/eda_overview.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117', edgecolor='none')
plt.show()
print("✅ EDA complete")

## ⚙️ Step 4: Feature Engineering

This is where we go beyond basic models. We create **domain-driven financial features** that credit analysts actually use.

In [ ]:
def engineer_features(df):
    """Domain-driven financial feature engineering."""
    df = df.copy()

    # ── Temporal features ──────────────────────────────────────────────
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
    df['YEARS_EMPLOYED'] = np.where(
        df['DAYS_EMPLOYED'] == 365243, 0,
        -df['DAYS_EMPLOYED'] / 365
    )
    df['EMPLOYMENT_RATIO'] = df['YEARS_EMPLOYED'] / (df['AGE_YEARS'] + 1e-6)
    df['YEARS_REGISTRATION'] = -df['DAYS_REGISTRATION'] / 365
    df['YEARS_ID_PUBLISH'] = -df['DAYS_ID_PUBLISH'] / 365

    # ── Credit stress ratios (key credit bureau signals) ───────────────
    df['CREDIT_INCOME_RATIO']  = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['CREDIT_TERM']          = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + 1)
    df['GOODS_PRICE_RATIO']    = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)
    df['INCOME_PER_PERSON']    = df['AMT_INCOME_TOTAL'] / (df['CNT_CHILDREN'] + 1)

    # ── External score aggregations (credit bureau composite) ──────────
    ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
                if c in df.columns]
    if ext_cols:
        df['EXT_SOURCE_MEAN']   = df[ext_cols].mean(axis=1)
        df['EXT_SOURCE_MIN']    = df[ext_cols].min(axis=1)
        df['EXT_SOURCE_STD']    = df[ext_cols].std(axis=1).fillna(0)
        df['EXT_SOURCE_PROD']   = df[ext_cols].prod(axis=1)
        # Weighted composite: EXT_SOURCE_2 has highest predictive power
        weights = {'EXT_SOURCE_1': 0.2, 'EXT_SOURCE_2': 0.5, 'EXT_SOURCE_3': 0.3}
        df['EXT_WEIGHTED'] = sum(
            df[c].fillna(df[c].median()) * w
            for c, w in weights.items() if c in df.columns
        )

    # ── Risk flags ─────────────────────────────────────────────────────
    df['HIGH_CREDIT_STRESS'] = (df['CREDIT_INCOME_RATIO'] > 5).astype(int)
    df['UNEMPLOYED_FLAG']    = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['YOUNG_BORROWER']     = (df['AGE_YEARS'] < 27).astype(int)
    df['HIGH_ANNUITY_RATIO'] = (df['ANNUITY_INCOME_RATIO'] > 0.3).astype(int)

    # ── Social circle risk contagion ───────────────────────────────────
    if 'OBS_30_CNT_SOCIAL_CIRCLE' in df.columns and 'DEF_30_CNT_SOCIAL_CIRCLE' in df.columns:
        df['SOCIAL_DEFAULT_RATIO'] = (
            df['DEF_30_CNT_SOCIAL_CIRCLE'] /
            (df['OBS_30_CNT_SOCIAL_CIRCLE'] + 1)
        )

    # ── Encode categoricals ────────────────────────────────────────────
    binary_map = {'Y': 1, 'N': 0, 'M': 1, 'F': 0, 'XNA': -1}
    for col in ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CODE_GENDER']:
        if col in df.columns:
            df[col] = df[col].map(binary_map).fillna(0)

    if 'NAME_EDUCATION_TYPE' in df.columns:
        edu_order = {
            'Lower secondary': 0,
            'Secondary / secondary special': 1,
            'Incomplete higher': 2,
            'Higher education': 3,
            'Academic degree': 4
        }
        df['EDUCATION_ORDINAL'] = df['NAME_EDUCATION_TYPE'].map(edu_order).fillna(1)

    if 'NAME_CONTRACT_TYPE' in df.columns:
        df['IS_REVOLVING'] = (df['NAME_CONTRACT_TYPE'] == 'Revolving loans').astype(int)

    return df


df_eng = engineer_features(df_raw)

# Select final feature set
FEATURE_COLS = [
    # Original numerical
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'REGION_RATING_CLIENT', 'HOUR_APPR_PROCESS_START', 'CNT_CHILDREN',
    'FLAG_DOCUMENT_3', 'AMT_REQ_CREDIT_BUREAU_YEAR',
    # Engineered
    'AGE_YEARS', 'YEARS_EMPLOYED', 'EMPLOYMENT_RATIO',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
    'GOODS_PRICE_RATIO', 'INCOME_PER_PERSON',
    'EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN', 'EXT_SOURCE_STD',
    'EXT_SOURCE_PROD', 'EXT_WEIGHTED',
    'HIGH_CREDIT_STRESS', 'UNEMPLOYED_FLAG', 'YOUNG_BORROWER',
    'HIGH_ANNUITY_RATIO', 'SOCIAL_DEFAULT_RATIO',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CODE_GENDER',
    'EDUCATION_ORDINAL', 'IS_REVOLVING',
]
# Only keep columns that exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_eng.columns]
TARGET_COL = 'TARGET'

print(f"✅ Engineered features: {len(FEATURE_COLS)} total")
print(f"Original numeric columns used: {len([c for c in FEATURE_COLS if c in df_raw.columns])}")
print(f"New engineered columns: {len([c for c in FEATURE_COLS if c not in df_raw.columns])}")

## 🔧 Step 5: Data Preprocessing & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

X = df_eng[FEATURE_COLS]
y = df_eng[TARGET_COL]

# Stratified split — preserves class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessing pipeline: median imputation + robust scaling
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())   # robust to outliers — critical for financial data
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

# Class imbalance ratio (for class_weight)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}")
print(f"Class imbalance ratio: {pos_weight:.1f}:1  (neg:pos)")
print(f"Features: {X_train.shape[1]}")

## 🤖 Step 6: Model Comparison

We compare 4 meaningful models — each with a different bias/variance profile. Scored on **ROC-AUC** and **Average Precision** (better for imbalanced classes).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve)
import time

models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=500, random_state=42, C=0.1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced',
        max_depth=8, min_samples_leaf=50,
        random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.05,
        max_depth=4, subsample=0.8, random_state=42
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200, learning_rate=0.05,
        num_leaves=31, class_weight='balanced',
        random_state=42, n_jobs=-1, verbose=-1
    ),
}

results = {}
trained_models = {}

print(f"{'Model':<25} {'ROC-AUC':>10} {'Avg Prec':>10} {'Time(s)':>10}")
print("-" * 60)

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_prep, y_train)
    elapsed = time.time() - t0

    y_prob = model.predict_proba(X_test_prep)[:, 1]
    roc    = roc_auc_score(y_test, y_prob)
    ap     = average_precision_score(y_test, y_prob)

    results[name] = {'roc_auc': roc, 'avg_precision': ap, 'time': elapsed, 'probs': y_prob}
    trained_models[name] = model

    print(f"{name:<25} {roc:>10.4f} {ap:>10.4f} {elapsed:>10.1f}")

best_model_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f"\n🏆 Best model: {best_model_name}  (ROC-AUC: {results[best_model_name]['roc_auc']:.4f})")

In [ ]:
# Plot ROC & PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0f1117')
colors_plot = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for ax in axes:
    ax.set_facecolor('#1a1d2e')

for (name, res), color in zip(results.items(), colors_plot):
    fpr, tpr, _ = roc_curve(y_test, res['probs'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f"{name} ({res['roc_auc']:.3f})")

axes[0].plot([0,1],[0,1],'--', color='#555555', lw=1)
axes[0].set_xlabel('False Positive Rate', color='#aaaaaa')
axes[0].set_ylabel('True Positive Rate', color='#aaaaaa')
axes[0].set_title('ROC Curves — Model Comparison', color='white', fontweight='bold')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(True, alpha=0.2)

for (name, res), color in zip(results.items(), colors_plot):
    prec, rec, _ = precision_recall_curve(y_test, res['probs'])
    axes[1].plot(rec, prec, color=color, lw=2,
                 label=f"{name} (AP={res['avg_precision']:.3f})")

baseline = y_test.mean()
axes[1].axhline(baseline, color='#555555', lw=1, ls='--',
                label=f'Baseline ({baseline:.3f})')
axes[1].set_xlabel('Recall', color='#aaaaaa')
axes[1].set_ylabel('Precision', color='#aaaaaa')
axes[1].set_title('Precision-Recall Curves', color='white', fontweight='bold')
axes[1].legend(fontsize=9, loc='upper right')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('/content/model_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

## 🎯 Step 7: Hyperparameter Tuning with Optuna

We tune the best model (LightGBM) using Bayesian optimization — far more efficient than GridSearch.

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 400),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves':      trial.suggest_int('num_leaves', 15, 63),
        'max_depth':       trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'class_weight': 'balanced',
        'random_state': 42,
        'verbose': -1,
        'n_jobs': -1,
    }
    model = LGBMClassifier(**params)
    # 3-fold CV on training data
    scores = cross_val_score(
        model, X_train_prep, y_train,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    )
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, timeout=240,
               show_progress_bar=True)

print(f"\n✅ Best CV ROC-AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Train final tuned model
best_params = study.best_params
best_params.update({'class_weight': 'balanced', 'random_state': 42, 'verbose': -1, 'n_jobs': -1})

final_model = LGBMClassifier(**best_params)
final_model.fit(X_train_prep, y_train)

y_prob_final = final_model.predict_proba(X_test_prep)[:, 1]
roc_final = roc_auc_score(y_test, y_prob_final)
ap_final  = average_precision_score(y_test, y_prob_final)

baseline_roc = results['LightGBM']['roc_auc']
print(f"\nTuned LightGBM — ROC-AUC: {roc_final:.4f}  (baseline: {baseline_roc:.4f}, +{roc_final-baseline_roc:.4f})")
print(f"Tuned LightGBM — Avg Precision: {ap_final:.4f}")

## 📐 Step 8: Probability Calibration

Raw ML probabilities are often poorly calibrated — this matters in credit risk where we need **reliable probability estimates**, not just rankings.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Calibrate using isotonic regression (more flexible than Platt scaling for large n)
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_test_prep, y_test)  # calibrate on held-out set

y_prob_cal = calibrated_model.predict_proba(X_test_prep)[:, 1]

# Compare calibration curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d2e')

for probs, name, color in [
    (y_prob_final, 'Uncalibrated', '#FF6B6B'),
    (y_prob_cal,   'Calibrated',   '#4ECDC4'),
]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10)
    axes[0].plot(mean_pred, frac_pos, 's-', color=color, lw=2, label=name)

axes[0].plot([0,1],[0,1],'--', color='#888888', label='Perfect calibration')
axes[0].set_xlabel('Mean predicted probability', color='#aaaaaa')
axes[0].set_ylabel('Fraction of positives', color='#aaaaaa')
axes[0].set_title('Calibration Curve (Reliability Diagram)', color='white', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.2)

# Probability distribution
for t, color in zip([0, 1], ['#4CAF50', '#FF5252']):
    mask = y_test == t
    axes[1].hist(y_prob_cal[mask], bins=40, alpha=0.6, color=color, density=True,
                 label='No Default' if t == 0 else 'Default')
axes[1].set_xlabel('Predicted Default Probability', color='#aaaaaa')
axes[1].set_ylabel('Density', color='#aaaaaa')
axes[1].set_title('Score Separation by Class', color='white', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('/content/calibration.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(f"Calibrated ROC-AUC: {roc_auc_score(y_test, y_prob_cal):.4f}")

## 💰 Step 9: Business-Aware Threshold Optimization

**The critical insight**: In credit risk, a false negative (approving a defaulter) costs ~10× more than a false positive (rejecting a good borrower). We optimize the threshold using realistic cost assumptions.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Business cost parameters
COST_FN = 10   # Cost of approving a defaulter (loan loss)
COST_FP = 1    # Cost of rejecting a good borrower (lost revenue opportunity)

thresholds = np.linspace(0.01, 0.99, 200)
costs = []

for thresh in thresholds:
    y_pred_t = (y_prob_cal >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    total_cost = COST_FN * fn + COST_FP * fp
    costs.append(total_cost)

optimal_idx   = np.argmin(costs)
OPTIMAL_THRESHOLD = thresholds[optimal_idx]

# Plot cost curve
fig, ax = plt.subplots(figsize=(10, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d2e')
ax.plot(thresholds, costs, color='#4ECDC4', lw=2)
ax.axvline(OPTIMAL_THRESHOLD, color='#FF6B6B', lw=2, ls='--',
           label=f'Optimal threshold = {OPTIMAL_THRESHOLD:.3f}')
ax.axvline(0.5, color='#888888', lw=1, ls=':', label='Default 0.5 threshold')
ax.fill_between(thresholds, costs, alpha=0.15, color='#4ECDC4')
ax.set_xlabel('Classification Threshold', color='#aaaaaa')
ax.set_ylabel(f'Business Cost  (FN×{COST_FN} + FP×{COST_FP})', color='#aaaaaa')
ax.set_title('Cost-Optimal Threshold Selection', color='white', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('/content/threshold_optimization.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

# Final classification report at optimal threshold
y_pred_opt = (y_prob_cal >= OPTIMAL_THRESHOLD).astype(int)
print(f"\n📊 Classification Report at Optimal Threshold ({OPTIMAL_THRESHOLD:.3f})")
print(classification_report(y_test, y_pred_opt, target_names=['No Default', 'Default']))
print(f"Business cost at optimal threshold: {costs[optimal_idx]:,.0f}")
print(f"Business cost at threshold=0.5:     {costs[np.argmin(np.abs(thresholds-0.5))]:,.0f}")

## 🧠 Step 10: SHAP Explainability

SHAP (SHapley Additive exPlanations) answers: **why did the model make this decision?** This is essential for regulatory compliance in credit scoring (Equal Credit Opportunity Act).

In [ ]:
import shap
shap.initjs()

# SHAP TreeExplainer is exact and fast for tree models
explainer = shap.TreeExplainer(final_model)

# Use a sample for speed
N_SHAP = min(2000, len(X_test_prep))
idx_sample = np.random.choice(len(X_test_prep), N_SHAP, replace=False)
X_shap = X_test_prep[idx_sample]

shap_values = explainer.shap_values(X_shap)
# LightGBM returns list for binary; take class-1
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f"✅ SHAP values computed for {N_SHAP} samples")

In [ ]:
# Summary plot — beeswarm
plt.figure(figsize=(10, 8), facecolor='#0f1117')
shap.summary_plot(
    shap_vals, X_shap,
    feature_names=FEATURE_COLS,
    plot_type='dot',
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance — Top 20 Features', color='white',
          fontweight='bold', fontsize=13, pad=15)
plt.gca().set_facecolor('#0f1117')
plt.savefig('/content/shap_summary.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

# Bar plot — mean absolute SHAP
plt.figure(figsize=(10, 7), facecolor='#0f1117')
shap.summary_plot(
    shap_vals, X_shap,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('Mean |SHAP| — Global Feature Importance', color='white',
          fontweight='bold', fontsize=13, pad=15)
plt.gca().set_facecolor('#0f1117')
plt.savefig('/content/shap_bar.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 💾 Step 11: Save All Artifacts

In [ ]:
import os
os.makedirs('/content/artifacts', exist_ok=True)

# Save model, preprocessor, metadata
joblib.dump(calibrated_model, '/content/artifacts/model_calibrated.pkl')
joblib.dump(preprocessor,     '/content/artifacts/preprocessor.pkl')
joblib.dump(SHAP_EXPLAINER := explainer, '/content/artifacts/shap_explainer.pkl')

import json
metadata = {
    'feature_cols': FEATURE_COLS,
    'optimal_threshold': float(OPTIMAL_THRESHOLD),
    'model_roc_auc': float(roc_auc_score(y_test, y_prob_cal)),
    'model_avg_precision': float(average_precision_score(y_test, y_prob_cal)),
    'best_params': best_params,
    'cost_fn': COST_FN,
    'cost_fp': COST_FP,
}
with open('/content/artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Download all artifacts
import shutil
shutil.make_archive('/content/credit_risk_artifacts', 'zip', '/content/artifacts')

from google.colab import files
files.download('/content/credit_risk_artifacts.zip')

print("✅ All artifacts saved and downloading!")
print("\nFiles saved:")
for f in os.listdir('/content/artifacts'):
    size = os.path.getsize(f'/content/artifacts/{f}') / 1024
    print(f"  {f}: {size:.1f} KB")

## 📊 Step 12: Final Performance Summary

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

print("=" * 60)
print("       CREDIT RISK INTELLIGENCE — FINAL RESULTS")
print("=" * 60)
print(f"Dataset:          Home Credit Default Risk")
print(f"Training samples: {len(y_train):,}")
print(f"Test samples:     {len(y_test):,}")
print(f"Features:         {len(FEATURE_COLS)} (incl. {len([c for c in FEATURE_COLS if c not in df_raw.columns])} engineered)")
print()
print("── Model Performance ──")
print(f"ROC-AUC:          {roc_auc_score(y_test, y_prob_cal):.4f}")
print(f"Average Precision:{average_precision_score(y_test, y_prob_cal):.4f}")
print(f"Brier Score:      {brier_score_loss(y_test, y_prob_cal):.4f}  (lower=better)")
print()
print("── Business Optimization ──")
print(f"Optimal Threshold:{OPTIMAL_THRESHOLD:.3f}")
print(f"Cost weights:     FN={COST_FN}x, FP={COST_FP}x")
print()
print("── Pipeline ──")
print("  Imputation:     Median (robust to outliers)")
print("  Scaling:        RobustScaler")
print("  Model:          LightGBM + Isotonic Calibration")
print("  Tuning:         Optuna Bayesian (30 trials, 3-fold CV)")
print("  Explainability: SHAP TreeExplainer")
print("=" * 60)